# VRI 2026 Model Training on Google Colab

This notebook installs dependencies, clones the latest code from GitHub, and trains our YOLO and ResNet18 models using Google Colab GPUs.

## 1. Setup & Dependencies
Run this cell to clone the repository and install dependencies.

In [ ]:
%cd /content
!git clone https://github.com/Jack0468/ball_balance_video_controlled.git
!pip install ultralytics opencv-python-headless pandas numpy albumentations
import torch
print(f"Setup complete. Using torch {torch.__version__} ({torch.cuda.get_device_properties(0).name if torch.cuda.is_available() else 'CPU'})")

## 2. Upload Your Data
On your local PC, run `data_processing/generate_yolo_labels_from_images.py` to generate the `labels` folder.

Then, **Zip** your `images/` folder, `labels/` folder, and `labels.csv` file into one zip file and upload it to your Google Drive root directory.

Run the cell below to extract it directly into `/content/data`:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/data
!unzip -q -o /content/drive/MyDrive/vri_2026_data.zip -d /content/data/

## 3. Train YOLO Marker & Ball Detector
We set the `--project` flag to save weights DIRECTLY to your Google Drive, so nothing is lost if Colab disconnects!

In [ ]:
%cd /content/ball_balance_video_controlled/host_software/ml_vision

# Note: Colab usually runs Linux, so we use forward slashes for paths
!python training/train_yolo.py --data_dir /content/data --epochs 100 --batch_size 16 project=/content/drive/MyDrive/VRI_2026_Trained_Models/

## 4. Train ResNet18 Expert Tracker
Run this to train the ResNet model. Weights are saved in `/content/`, but we added a backup step below.

In [ ]:
%cd /content/ball_balance_video_controlled/host_software/ml_vision
!python training/train_expert_tracker.py --data_dir /content/data

## 5. Export ResNet Weights
Once ResNet training is finished, run this cell to copy the newly generated weights back to your Google Drive so you can download them to your PC.

In [ ]:
!cp -r /content/ball_balance_video_controlled/host_software/ml_vision/models /content/drive/MyDrive/VRI_2026_Trained_Models/